## Phase 3 — Hunyuan3D Mesh Generation

**Input:** Rendered image (PNG) — upload when prompted, or place in `/content/input/`.  
**Output:** GLB mesh file in `/content/mesh_results/`.  

**How to run:** Execute all cells top to bottom. Model weights (~25 GB) are cached to Google Drive on first run. Requires a GPU runtime.

In [ ]:
import os
import sys

try:
    import google.colab
    IN_COLAB = True
    from google.colab import files, drive
    print('Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('WARNING: This notebook is designed for Google Colab')

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/3D_Models_Cache'
    os.makedirs(DRIVE_BASE, exist_ok=True)
    print(f'Drive mounted. Cache directory: {DRIVE_BASE}')
else:
    DRIVE_BASE = './models_cache'
    os.makedirs(DRIVE_BASE, exist_ok=True)

In [ ]:
import subprocess
import sys

print('Installing required packages...')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'Pillow'], capture_output=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rembg[gpu]'], capture_output=True)

packages = [
    'trimesh',
    'omegaconf',
    'einops',
    'huggingface_hub',
    'safetensors',
    'accelerate',
    'pymeshlab',
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], capture_output=True)

print('Dependencies installed!')

In [ ]:
import os
import sys
import subprocess
import gc
import shutil
import types
import importlib.util
import urllib.request
import zipfile
import io
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
from huggingface_hub import snapshot_download

try:
    import google.colab
    IN_COLAB = True
    from google.colab import files, drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DRIVE_BASE = '/content/drive/MyDrive/3D_Models_Cache'
    INPUT_DIR = '/content/input'
    OUTPUT_DIR = '/content/mesh_results'
else:
    DRIVE_BASE = './models_cache'
    INPUT_DIR = './input'
    OUTPUT_DIR = './mesh_results'

WEIGHTS_DIR = os.path.join(DRIVE_BASE, 'weights')

HY_DIR = os.path.join(WEIGHTS_DIR, 'hunyuan3d-2mini')
HY3DGEN_DIR = os.path.join(WEIGHTS_DIR, 'hy3dgen')

for d in [INPUT_DIR, OUTPUT_DIR, WEIGHTS_DIR]:
    os.makedirs(d, exist_ok=True)

try:
    import rembg
    REMBG_AVAILABLE = True
    print('rembg available for background removal')
except ImportError:
    REMBG_AVAILABLE = False
    print('rembg not available, using simple background removal')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def outpath(model_name, ext='glb'):
    return os.path.join(OUTPUT_DIR, f'{model_name}_output.{ext}')

def prep_image(path):
    img = Image.open(path).convert('RGBA')
    if REMBG_AVAILABLE:
        try:
            return rembg.remove(img)
        except Exception as e:
            print(f'  rembg failed: {e}, using fallback')
    data = np.array(img)
    mask = (data[:,:,0] > 240) & (data[:,:,1] > 240) & (data[:,:,2] > 240)
    data[mask, 3] = 0
    return Image.fromarray(data)

def report(path):
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f'  Saved: {os.path.basename(path)} ({size_kb:.1f} KB)')
    else:
        print(f'  ERROR: Failed to save {path}')

class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

def download_url_with_progress(url, desc='Downloading'):
    with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=desc) as t:
        data, _ = urllib.request.urlretrieve(url, reporthook=t.update_to)
        return open(data, 'rb').read()

print('\nLoading input image...')
INPUT_IMAGE = os.path.join(INPUT_DIR, 'input_image.png')

existing_images = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))] if os.path.exists(INPUT_DIR) else []

if existing_images:
    INPUT_IMAGE = os.path.join(INPUT_DIR, existing_images[0])
    print(f'Using existing image: {INPUT_IMAGE}')
else:
    print('Please upload your input image:')
    if IN_COLAB:
        uploaded = files.upload()
        for fname, content in uploaded.items():
            INPUT_IMAGE = os.path.join(INPUT_DIR, fname)
            with open(INPUT_IMAGE, 'wb') as f:
                f.write(content)
            print(f'Saved: {INPUT_IMAGE}')
            break
    else:
        INPUT_IMAGE = input('Enter path to your image: ')

print(f'\nInput image: {INPUT_IMAGE}')
img = Image.open(INPUT_IMAGE)
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.axis('off')
plt.title('Input Image')
plt.show()

print(f'\nOutput directory: {OUTPUT_DIR}')
print(f'Weights cache: {WEIGHTS_DIR}')
print('\nSetup complete!')

In [ ]:
print('=== Checking/Downloading Hunyuan3D (~25GB) ===')
if os.path.exists(HY_DIR) and len(os.listdir(HY_DIR)) > 5:
    print('  Hunyuan3D weights: Already in Drive')
else:
    print('  Downloading Hunyuan3D weights...')
    snapshot_download(
        repo_id='tencent/Hunyuan3D-2mini',
        local_dir=HY_DIR,
        ignore_patterns=['*.md', 'LICENSE', 'NOTICE']
    )
    print('  Hunyuan3D weights: Downloaded')

if os.path.exists(os.path.join(HY3DGEN_DIR, 'hy3dgen')):
    print('  hy3dgen code: Already in Drive')
else:
    print('  Downloading hy3dgen code...')
    url = 'https://github.com/Tencent/Hunyuan3D-2/archive/refs/heads/main.zip'
    data = download_url_with_progress(url, desc='  hy3dgen')
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        for m in zf.namelist():
            if not m.startswith('Hunyuan3D-2-main/hy3dgen/'):
                continue
            rel = m[len('Hunyuan3D-2-main/'):]
            tgt = os.path.join(HY3DGEN_DIR, rel)
            if m.endswith('/'):
                os.makedirs(tgt, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(tgt), exist_ok=True)
                with open(tgt, 'wb') as f:
                    f.write(zf.read(m))
    print('  hy3dgen code: Downloaded')

print('\n Hunyuan3D model ready!')

In [ ]:
print('=== Hunyuan3D 2 Mini ===')

if 'HY3DGEN_DIR' not in dir():
    raise RuntimeError("Please run Cell 3 (Setup Paths & Load Image) first!")

if HY3DGEN_DIR not in sys.path:
    sys.path.insert(0, HY3DGEN_DIR)

from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

print('Loading Hunyuan3D pipeline...')
dtype = torch.float16 if DEVICE == 'cuda' else torch.float32
pipe_hunyuan = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    HY_DIR,
    subfolder='hunyuan3d-dit-v2-mini',
    use_safetensors=True,
    device=DEVICE,
    dtype=dtype
)

print('Processing image...')
img = prep_image(INPUT_IMAGE)

with torch.no_grad():
    output = pipe_hunyuan(
        image=img,
        num_inference_steps=30,
        octree_resolution=380,
        guidance_scale=5.5,
        num_chunks=4000,
        output_type='trimesh'
    )

mesh = output[0]
dst = outpath('hunyuan3d')
mesh.export(dst)
report(dst)

del mesh, output, pipe_hunyuan
free()
print('Hunyuan3D done!')

In [ ]:
print('=== Generated Mesh ===')
mesh_files = []
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith(('.glb', '.obj', '.ply')):
        path = os.path.join(OUTPUT_DIR, f)
        size_kb = os.path.getsize(path) / 1024
        print(f'  {f} ({size_kb:.1f} KB)')
        mesh_files.append(f)

print(f'\nTotal: {len(mesh_files)} mesh file(s)')

ZIP_NAME = 'hunyuan3d_result'
print(f'\nCreating {ZIP_NAME}.zip...')
shutil.make_archive(ZIP_NAME, 'zip', OUTPUT_DIR)
zip_size = os.path.getsize(f'{ZIP_NAME}.zip') / (1024 * 1024)
print(f'Created: {ZIP_NAME}.zip ({zip_size:.2f} MB)')

if IN_COLAB:
    print('\nStarting download...')
    files.download(f'{ZIP_NAME}.zip')
else:
    print(f'\nResults saved to: {OUTPUT_DIR}')
    print(f'Zip file: {ZIP_NAME}.zip')

print('\n Done! Hunyuan3D completed.')